# POL SAOCOM demo

In [ ]:
"""
File Name: pol.ipynb

Authors:
    Juan M. Lopez-Sanchez, Alejandro Mestre-Quereda
    University of Alicante

Date: June 2025

Description:
------------
This script performs the generation of POL products at a single date
from dual-pol SAOCOM images in CSLC format.
Options and configuration parameters about some processing steps can be set
by the user.

Dependencies:
-------------
- os
- numpy
- rosel

Usage:
------
To use this script, define the following parameters:

    - common_input_folder: Path to the main input folder with preprocessed CLSC data
    - common_output_folder: Path to the main output folder in which the POL products will be saved
    - site: 'Naples'', 'Salamanca', ''BlackForest' or 'Brest'
    - orbit: 'ASC' or 'DSC'
    - beam: 'S3' or 'S4'
    - estimation_type = Select one option: 'MULTI-LOOK', 'BOXCAR' or 'SHIROMA'
    - kernel_size = [ 9, 5 ] # [azimuth,range] Required for 'MULTI-LOOK' and 'BOXCAR'
    - acquisition_date: One of the timestamps listed in the cslc['acquisition_date'] variable, e.g. '2024-08-23T06:10:49.955322'

"""

In [1]:
import os
import numpy as np

import rosel.io
import rosel.computation
import rosel.shiroma

######################################
# PARAMETERS
######################################

common_input_folder = "/media/s3-ard-rose-l/SAOCOM"
common_output_folder = "/media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR/POL"

# Site, orbit pass, and beam

site = "Salamanca"  # Naples, Salamanca, BlackForest, Brest
orbit = "ASC"  # ASC, DSC
beam = "S4"  # S3, S4

# Processing options

estimation_type = "BOXCAR"  # 'MULTI-LOOK' or 'BOXCAR' or 'SHIROMA'

kernel_size = [9, 5]  # [azimuth,range] Required for 'MULTI-LOOK' and 'BOXCAR'

# %%###################################
# MAIN PROGRAM
######################################

if __name__ == "__main__":

    # Input folder with pre-processsed images in CSLC format

    input_folder = os.path.join(common_input_folder, site, orbit, beam)
    in_path = rosel.io.normpath(input_folder)

    # Output folder

    common_out_path = os.path.join(
        common_output_folder, site, orbit, beam, estimation_type
    )

    if estimation_type != "SHIROMA":
        common_out_path = os.path.join(
            common_out_path, str(kernel_size[0]) + "x" + str(kernel_size[1])
        )

In [2]:
subfolder_list = [
    os.path.basename(f.path) for f in os.scandir(input_folder) if f.is_dir()
]

for subfolder in subfolder_list:

    if subfolder.endswith("_VV"):
        cslc_path_vv = os.path.join(in_path, subfolder, "coreg")

    if subfolder.endswith("_VH"):
        cslc_path_vh = os.path.join(in_path, subfolder, "coreg")

    if subfolder == "SHIROMA":
        shiroma_path = os.path.join(in_path, "SHIROMA")

selection_path = os.path.join(shiroma_path, "selection")
global_values_path = os.path.join(shiroma_path, "global_values")
rtc_ap_path = os.path.join(shiroma_path, "rtc_ap")

########################
# PROCESSING
########################

In [3]:
# Check the available acquistions to select one:

cslc = rosel.io.read(cslc_path_vv, consolidate_metadata=False)
cslc["acquisition_date"]

<xarray.DataArray 'acquisition_date' (acquisition_date: 13)> Size: 1kB
array(['2023-04-07T06:07:07.151076', '2023-07-04T06:10:52.256791',
       '2023-08-05T06:10:53.968824', '2023-09-06T06:10:54.524005',
       '2023-10-08T06:10:53.810033', '2023-11-09T06:10:52.719348',
       '2024-02-13T06:10:51.872102', '2024-07-22T06:10:52.105407',
       '2024-08-23T06:10:49.955322', '2024-09-24T06:10:51.209735',
       '2024-10-26T06:10:56.417370', '2024-12-29T06:10:52.380350',
       '2025-01-30T06:10:50.541658'], dtype='<U26')
Coordinates:
  * acquisition_date  (acquisition_date) <U26 1kB '2023-04-07T06:07:07.151076...

In [4]:
# The user needs to select one of the dates from the list.
# For instance:

acquisition_date = "2024-09-24T06:10:51.209735"  # ASC/S4 Salamanca

date_txt = acquisition_date[:10]  # Only the string with the date

print("Date: ", date_txt)

out_path = os.path.join(common_out_path, date_txt)

Date:  2024-09-24


In [5]:
print("Read SLC images...")

slc_vv = rosel.io.read(cslc_path_vv, consolidate_metadata=False).cstack.sel(
    acquisition_date=acquisition_date
)
slc_vh = rosel.io.read(cslc_path_vh, consolidate_metadata=False).cstack.sel(
    acquisition_date=acquisition_date
)

# Read auxiliary layers for flattening and geocoding

selection = rosel.io.read(selection_path, consolidate_metadata=False)
A = rosel.io.read(rtc_ap_path, mask_and_scale=False, consolidate_metadata=False).v

Read SLC images...


In [6]:
print("Flatten the SLC values of both channels ...")

slc_vv_flat = rosel.shiroma.flatten_radar_image(
    slc_vv, A, data_path=f"{out_path}/flattened_slc_vv", read=True
)

slc_vh_flat = rosel.shiroma.flatten_radar_image(
    slc_vh, A, data_path=f"{out_path}/flattened_slc_vh", read=True
)

Flatten the SLC values of both channels ...
2025-09-01 20:11:46 [warning  ] No global client found a new one will be created
2025-09-01 20:11:46 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=10023071398>
2025-09-01 20:11:48 [debug    ] Selected scatter strategy: once-and-for-all
2025-09-01 20:11:48 [debug    ] Number of blocks to be processed: 216


2025-09-01 20:18:22,015 - distributed.scheduler - ERROR - Removing worker 'tcp://127.0.0.1:38889' caused the cluster to lose scattered data, which can't be recovered: {'DataArray-e34ee6296f31567f334e39463452e71d', 'float-a4f3c38d91658f7a6c8e3611907bf266', 'DataArray-4c8378c2ba0f39325905f663a930437e', 'str-e0d6c35ac840b495e8dc088eb90455c8'} (stimulus_id='handle-worker-cleanup-1756750702.0158544')


2025-09-01 20:18:55 [warning  ] No global client found a new one will be created
2025-09-01 20:18:55 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=10023071398>
2025-09-01 20:18:56 [debug    ] Selected scatter strategy: once-and-for-all
2025-09-01 20:18:56 [debug    ] Number of blocks to be processed: 216


2025-09-01 20:25:30,628 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:42023' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'execute-b17b5ad7-c99c-4846-bd44-7b0add9165bf', 'execute-53ea841e-b7e4-442d-954c-85260fab593c', 'execute-9bd3c521-3818-4feb-aa45-a70aa57004bc', 'execute-fc875ddc-b0fc-44f1-acef-d7a1d129ab59', 'execute-6554bbe4-d6bb-4093-9436-69555be94111', 'execute-04c4c2dc-c222-4ef4-b53c-de18a6637269', 'execute-796f539e-233e-49ee-9a49-0442d4c285d8', 'execute-17f4f6db-943b-49f6-8280-83a115649bb2', 'execute-43cccb14-6eca-46d4-9d22-d0ceabcdb920', 'execute-e7c2f2f6-3378-4184-ba37-7aec6a945406'} (stimulus_id='handle-worker-cleanup-1756751130.6287313')
2025-09-01 20:25:30,639 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:34399' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {'execute-df324fb2-a5e1-4536-b385-f71cc38c98b5', 'execute-bfa89142-befb-4d78-b64d-d7

In [7]:
print("Covariance matrix formation: cross-products and estimation ...")

if estimation_type == "MULTI-LOOK":

    multilook = {"azimuth_time": kernel_size[0], "slant_range_time": kernel_size[1]}

    C11 = (
        np.abs((slc_vv_flat * slc_vv_flat.conj()))
        .astype(np.float32)
        .coarsen(
            azimuth_time=multilook["azimuth_time"],
            slant_range_time=multilook["slant_range_time"],
            boundary="pad",
        )
        .mean()
        .chunk(azimuth_time=300, slant_range_time=300)
    )

    C22 = (
        np.abs((slc_vh_flat * slc_vh_flat.conj()))
        .astype(np.float32)
        .coarsen(
            azimuth_time=multilook["azimuth_time"],
            slant_range_time=multilook["slant_range_time"],
            boundary="pad",
        )
        .mean()
        .chunk(azimuth_time=300, slant_range_time=300)
    )

    C12 = (
        (slc_vv_flat * slc_vh_flat.conj())
        .coarsen(
            azimuth_time=multilook["azimuth_time"],
            slant_range_time=multilook["slant_range_time"],
            boundary="pad",
        )
        .mean()
        .chunk(azimuth_time=300, slant_range_time=300)
    )

elif estimation_type == "BOXCAR":

    boxcar_kernel = {"azimuth_time": kernel_size[0], "slant_range_time": kernel_size[1]}

    data = np.abs((slc_vv_flat * slc_vv_flat.conj())).astype(np.float32)
    C11 = (
        data.rolling(
            azimuth_time=boxcar_kernel["azimuth_time"],
            slant_range_time=boxcar_kernel["slant_range_time"],
            center=True,
        )
        .mean()
        .chunk({"azimuth_time": 1024, "slant_range_time": 1024})
    )

    data = np.abs((slc_vh_flat * slc_vh_flat.conj())).astype(np.float32)
    C22 = (
        data.rolling(
            azimuth_time=boxcar_kernel["azimuth_time"],
            slant_range_time=boxcar_kernel["slant_range_time"],
            center=True,
        )
        .mean()
        .chunk({"azimuth_time": 1024, "slant_range_time": 1024})
    )

    data = slc_vv_flat * slc_vh_flat.conj()
    C12 = (
        data.rolling(
            azimuth_time=boxcar_kernel["azimuth_time"],
            slant_range_time=boxcar_kernel["slant_range_time"],
            center=True,
        )
        .mean()
        .chunk({"azimuth_time": 1024, "slant_range_time": 1024})
    )

elif estimation_type == "SHIROMA":

    C11 = np.abs((slc_vv_flat * slc_vv_flat.conj())).astype(np.float32)

    C22 = np.abs((slc_vh_flat * slc_vh_flat.conj())).astype(np.float32)

    C12 = slc_vv_flat * slc_vh_flat.conj()

else:

    print("Wrong input for type of estimation of the COV matrix.")
    print("There are 3 options: 'MULTI-LOOK', 'BOXCAR' or 'SHIROMA'")

Covariance matrix formation: cross-products and estimation ...


In [8]:
print("Save COV entries ...")

rosel.io.write(C11.to_dataset(name="v"), f"{out_path}/COV/C11")
rosel.io.write(C22.to_dataset(name="v"), f"{out_path}/COV/C22")
rosel.io.write(C12.to_dataset(name="v"), f"{out_path}/COV/C12")

Save COV entries ...


<xarray.Dataset> Size: 2GB
Dimensions:  (azimuth_time: 26745, slant_range_time: 7835)
Dimensions without coordinates: azimuth_time, slant_range_time
Data variables:
    v        (azimuth_time, slant_range_time) complex64 2GB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
Attributes:
    obj_type:    xr.Dataset
    store_path:  /media/s3-ard-rose-l/PHASE_1/DEMONSTRATOR/POL/Salamanca/ASC/...

In [9]:
#   Geocoding is carried out using Shiroma's approach
#   In case of 'MULTI-LOOK' estimation, the output grid could be coarsened
#   Otherwise, the output grid posting is similar to SLC resolution

print("Geocoding ...")

if estimation_type == "MULTI-LOOK":

    gC11 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C11",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C11",
        multilook=multilook,
        read=True,
        enable_dask=True,
    )

    gC22 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C22",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C22",
        multilook=multilook,
        read=True,
        enable_dask=True,
    )

    gC12 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C12",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C12",
        multilook=multilook,
        read=True,
        enable_dask=True,
    )

else:

    gC11 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C11",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C11",
        read=True,
        enable_dask=True,
    )

    gC22 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C22",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C22",
        read=True,
        enable_dask=True,
    )

    gC12 = rosel.shiroma.map_geo_ap(
        c_path=f"{out_path}/COV/C12",
        simulation=selection,
        data_path_shiroma=shiroma_path,
        data_path_geo_ap=f"{out_path}/GCOV/C12",
        read=True,
        enable_dask=True,
    )

Geocoding ...
2025-09-01 20:48:06 [warning  ] No global client found a new one will be created
2025-09-01 20:48:06 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=10023071398>
2025-09-01 20:48:08 [debug    ] Selected scatter strategy: once-and-for-all
2025-09-01 20:48:08 [debug    ] Number of blocks to be processed: 1376


2025-09-01 21:19:32,625 - distributed.scheduler - ERROR - Removing worker 'tcp://127.0.0.1:44031' caused the cluster to lose scattered data, which can't be recovered: {'str-c6552332fa8e756d3b62bf6c01aaf051', 'int-9f254b8e860be77f10a5e1f4bc2e344b', 'ndarray-50018ab3709cf1d37b92f3cd2eb1ddf3', 'str-55148a9009bdb8a412ac3ef42a4da1f4', 'str-cf6ce1cf114f67c8fc57d66a249df66a', 'Dataset-8b1fc6e2a6dbd8af61d4c4190fdcdc7f', 'int-b134b498f0d07d794e5900e879e42c89'} (stimulus_id='handle-worker-cleanup-1756754372.6257036')


2025-09-01 21:21:32 [warning  ] No global client found a new one will be created
2025-09-01 21:21:32 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=10023071398>
2025-09-01 21:21:33 [debug    ] Selected scatter strategy: once-and-for-all
2025-09-01 21:21:33 [debug    ] Number of blocks to be processed: 1376


2025-09-01 21:52:49,545 - distributed.scheduler - ERROR - Removing worker 'tcp://127.0.0.1:37149' caused the cluster to lose scattered data, which can't be recovered: {'int-9f254b8e860be77f10a5e1f4bc2e344b', 'ndarray-50018ab3709cf1d37b92f3cd2eb1ddf3', 'str-5098d0b58e089ddf8c83a98769d98e9c', 'str-9d53394a83d1733bd364961b5a5c46db', 'str-cf6ce1cf114f67c8fc57d66a249df66a', 'Dataset-8b1fc6e2a6dbd8af61d4c4190fdcdc7f', 'int-b134b498f0d07d794e5900e879e42c89'} (stimulus_id='handle-worker-cleanup-1756756369.5449016')


2025-09-01 21:54:51 [warning  ] No global client found a new one will be created
2025-09-01 21:54:51 [debug    ] <Cluster info: type=LOCAL, n_jobs=16, job_memory=10023071398>
2025-09-01 21:54:52 [debug    ] Selected scatter strategy: once-and-for-all
2025-09-01 21:54:52 [debug    ] Number of blocks to be processed: 1376


2025-09-01 22:29:19,054 - distributed.scheduler - ERROR - Removing worker 'tcp://127.0.0.1:46251' caused the cluster to lose scattered data, which can't be recovered: {'str-22acf44cede15c84c724d8cbad6e5840', 'str-1b6b9e90f1e419dc6fe10283462194e7', 'ndarray-50018ab3709cf1d37b92f3cd2eb1ddf3', 'int-9f254b8e860be77f10a5e1f4bc2e344b', 'Dataset-8b1fc6e2a6dbd8af61d4c4190fdcdc7f', 'str-cf6ce1cf114f67c8fc57d66a249df66a', 'int-b134b498f0d07d794e5900e879e42c89'} (stimulus_id='handle-worker-cleanup-1756758559.0546377')
